Hyperparameter tuning - Fine tune the model (optimize the model for better output)

1. GridSearchCV - Try multiple combination
2. KerasClassificer - Keras model for sklearn
3. Pipeline - getting the output in structured workflow

In [1]:
## Import libraries

import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle


In [3]:
data = pd.read_csv('D:\GenAI\IntelliBI\GenAI\Class\Projects\GenAI\ANN-bank-customer\data\Churn_Modelling.csv')

<>:1: SyntaxWarning: invalid escape sequence '\G'
<>:1: SyntaxWarning: invalid escape sequence '\G'
C:\Users\Administrator\AppData\Local\Temp\ipykernel_13812\2940834260.py:1: SyntaxWarning: invalid escape sequence '\G'
  data = pd.read_csv('D:\GenAI\IntelliBI\GenAI\Class\Projects\GenAI\ANN-bank-customer\data\Churn_Modelling.csv')


In [4]:
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

In [5]:
## Map gender to 1 and 0
data['Gender'] = data['Gender'].map({'Male': 1, 'Female': 0})

In [6]:
onehot_geo = OneHotEncoder()
geo_encoded = onehot_geo.fit_transform(data[['Geography']]).toarray()

In [7]:
geo_encoded_df = pd.DataFrame(geo_encoded, 
                              columns=onehot_geo.get_feature_names_out(['Geography']))

In [8]:
data = pd.concat([data.drop(['Geography'], axis=1), geo_encoded_df], axis=1)

In [9]:
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [20]:
## Divide the data into features and target
X = data.drop(columns=['Exited'])
y = data['Exited']

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [23]:
X_train.shape, X_test.shape

((8000, 12), (2000, 12))

In [24]:

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Hyper parameter tuning

Write a function to create the model and try different parameters

- layer = 1 -> no loops
- layer = 2 -> add extra 1 layer
- layer = 3 -> add extra 2 layers

This will make the architecture more flexible

In [25]:
def create_model(neurons=32, layers =1):
    model = Sequential()  ## start te neural network
    model.add(Dense(neurons, activation='relu', input_shape=(X_train.shape[1],)))  ## add the first hidden layer with input shape

    ## Add more layers to this dynamically
    for _ in range(layers-1):   ## _ is a throwaway variable(python convention) used when we need to loop a specific number of times but dont know the exact value
        model.add(Dense(neurons, activation='relu'))  ## add more hidden layers
    
    model.add(Dense(1, activation='sigmoid'))  ## add the output layer
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return model

In [27]:
## Create a keras classifier

model = KerasClassifier(layers = 1, neurons=32, build_fn=create_model, verbose=1)

In [28]:
## Define the hyperparameter grid
param_grid = {
    'neurons': [16, 32, 64, 128],
    'layers': [1, 2, 3],
    'epochs': [50, 100, 150]
}

Based on the above parameter grids, we will try different combinations 

In [31]:
grid = GridSearchCV(estimator=model, 
                    param_grid=param_grid, 
                    n_jobs=-1, ## Uses all the CPU cores (faster)
                    cv=3,  ## 3-fold cross-validation, Data will split in 3 parts: training, validation, and test
                    verbose=1)  ## Shows the progress

In [ ]:
grid_result = grid.fit(X_train, y_train)

Fitting 3 folds for each of 36 candidates, totalling 108 fits


In [ ]:
print("Best: %f using %s" %(grid_result.best_score_, grid_result.best_params_))

In [ ]:
## New input data

new_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Female',
    'Age': 40,
    'Tenure': 3,
    'Balance': 50000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 60000,
}

In [ ]:
new_data_df = pd.DataFrame([new_data])

In [ ]:
with open('./encodings and scaling/geo_encoded.pkl', 'rb') as f:
    geo_encoded = pickle.load(f)

with open('./encodings and scaling/gender_encoded.pkl', 'rb') as f:
    gender_encoded = pickle.load(f)


In [ ]:
## Use the previous stored encoding and encode the Gender from new input data
new_data_df['Gender'] = gender_encoded.transform([[new_data_df['Gender']]]).toarray()

In [ ]:
geo_data = geo_encoded.transform([[new_data['Geography']]]).toarray()

In [ ]:
## Convert to dataframe

geo_df = pd.DataFrame(geo_data, columns=geo_encoded.get_feature_names_out(new_data_df['Geography']))

In [ ]:
geo_df

Gender = 0
Geography

Geography_France = 1
Geography_Germany = 0
Geography_Spain = 0

In [ ]:
new_data_df = pd.concat([new_data_df.drop('Geography', axis=1), geo_df], axis=1)

In [ ]:
new_data_df

In [ ]:
## Load the scaler

with open('./encodings and scaling/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)


In [ ]:
new_data_df_scaled = scaler.transform(new_data_df)

In [ ]:
new_data_df_scaled

Now our data is encoded and scaled using the same encodings and scalings which we have used to train our data. 
Basically it means our model accuracy will be better

prediction = model.predict([new_data_df_scaled])
prediction

In [ ]:
prediction_prob = prediction[0][0]



In [ ]:
## Decision logic 

if prediction_prob > 0.5:
    print("Customer is likely to churn.")
else:
    print("Customer is not likely to churn.")